# Using Walnuts from Python

This notebook will show to how run the same model (a simple standard normal)
implemented in Python, Numba (a just-in-time compiler package), and Stan.

In [1]:
import walnutpie

def summarize(name, fit):
    summarizer = walnutpie.Summarizer(fit)
    mean = summarizer.mean()
    std = summarizer.standard_deviation()
    ess = summarizer.ess()
    r_hat = summarizer.r_hat()
    draws = summarizer._stacked.shape[0]
    print(f"{name}\tdim\tmean\tstd\tess\trhat\tdraws")
    for i in range(len(mean)):
        print(
            f"\t{i}\t{mean[i]:.4f}\t{std[i]:.4f}\t{ess[i]:.2f}\t{r_hat[i]:.4f}\t{draws}"
        )


In [2]:
import os
import bridgestan

stan_code = os.path.join(
    bridgestan.compile.get_bridgestan_path(), "test_models/multi/multi.stan"
)
with open(stan_code, 'r') as f:
    print(f.read())

m = bridgestan.StanModel(
    stan_code,
    {"M": 2, "N": 0, "P": 0},
    make_args=["STAN_THREADS=1"],
)

data {
  int<lower=0> M;
  int<lower=0> N;
  int<lower=0> P;
}
parameters {
  vector[M] alpha;
}
model {
  alpha ~ normal(0, 1);
}



In [3]:
%%time
summarize("stan", walnutpie.walnuts_stan(m, seed=1234))

stan	dim	mean	std	ess	rhat	draws
	0	0.0193	0.9977	998.60	1.0007	923
	1	-0.0066	1.0066	726.85	1.0009	923
CPU times: user 8.72 ms, sys: 975 μs, total: 9.69 ms
Wall time: 3.72 ms


## Python
Defining a pure-python log density is simple and highly flexible, but will usually be slower than the other options due to the extra overhead of the Python language

In [4]:
import numpy as np
import scipy.stats


def logp(x):
    return np.sum(scipy.stats.norm.logpdf(x)), -x

In [5]:
%%time
summarize("pyfunc", walnutpie.walnuts_pyfunc(logp, num_params=2))

pyfunc	dim	mean	std	ess	rhat	draws
	0	-0.0088	1.0073	1114.36	1.0042	1325
	1	0.0005	0.9869	1155.30	1.0017	1325
CPU times: user 2.98 s, sys: 497 ms, total: 3.48 s
Wall time: 2.1 s


## Numba

If we are willing to use [`numba`](https://numba.pydata.org/), we can get much faster!

In [6]:
import numba
from numba import types
from numba_stats import norm


@numba.cfunc(
    types.intc(
        types.size_t,
        types.CPointer(types.double),
        types.CPointer(types.double),
        types.CPointer(types.double),
        types.voidptr,
    ),
    nopython=True,
)
def logp_numba(size, x_, grad_, lp, _):
    x = numba.carray(x_, size)
    lp[0] = norm.logpdf(x, 0.0, 1.0).sum()
    grad = numba.carray(grad_, size)
    grad[:] = -x
    return 0

In [7]:
%%time
summarize("numba", walnutpie.walnuts_pyfunc(logp_numba, num_params=2))

numba	dim	mean	std	ess	rhat	draws
	0	0.0062	0.9675	3363.08	1.0002	3869
	1	-0.0183	1.0031	2963.23	1.0001	3869
CPU times: user 21.8 ms, sys: 839 μs, total: 22.6 ms
Wall time: 7.41 ms
